# RAG pipeline evaluation

Visualizes the latest result of `python main.py eval` (`data/eval/latest_results.json`):

1. Retrieval metrics per mode (dense / bm25 / hybrid / hybrid_rerank).
2. Recall@5 per question category.
3. Ragas and abstention metrics for the final pipeline.

Dependencies: `pip install -r notebooks/requirements.txt`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

results = json.loads(Path('../data/eval/latest_results.json').read_text(encoding='utf-8'))
print(results['timestamp'], results['config'])

In [ ]:
summary = pd.DataFrame({mode: data['summary'] for mode, data in results['retrieval'].items()}).T
display(summary.style.format('{:.3f}').highlight_max(axis=0))

ax = summary[['hit@5', 'recall@5', 'mrr', 'ndcg@5']].plot.bar(figsize=(10, 4), rot=0)
ax.set_title('Retrieval by mode')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
plt.tight_layout()

In [ ]:
rows = [
    {'mode': mode, 'category': q['category'], 'recall@5': q['recall@5']}
    for mode, data in results['retrieval'].items()
    for q in data['per_question']
]
by_category = pd.DataFrame(rows).pivot_table(index='category', columns='mode', values='recall@5')
ax = by_category.plot.bar(figsize=(10, 4), rot=0)
ax.set_title('Recall@5 by category (exact identifiers show the contribution of BM25)')
ax.set_ylim(0, 1.05)
plt.tight_layout()

In [ ]:
print('Abstención:', results['generation']['abstention'])
if 'ragas' in results:
    ragas = pd.Series(results['ragas']['summary'])
    ax = ragas.plot.barh(figsize=(8, 3))
    ax.set_xlim(0, 1)
    ax.set_title('Ragas (hybrid_rerank)')
    plt.tight_layout()
    display(pd.DataFrame(results['ragas']['per_sample']))